# Processing Monitoring

Read-only overview of text counts for every configured dataset/model/split. The scan inspects existing CSV and Parquet artifacts and does not run inference.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from scripts.experiment_config import DATASETS, MODEL_IDS, build_datasets
from scripts.data_root import resolve_data_root
from processing_monitoring_utils import scan_processing_counts

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_rows", 300)

DATA_ROOT = resolve_data_root()
DATASETS = build_datasets(DATA_ROOT)
TEXT_DETECTION_DATA = DATA_ROOT

## Counts by Combination

In [ ]:
processing_counts = scan_processing_counts(DATA_ROOT, DATASETS, MODEL_IDS)

display(processing_counts)

stage_matrix = processing_counts.pivot_table(
    index=["dataset", "model", "split"],
    columns="stage",
    values="processed_texts",
    fill_value=0,
    observed=False,
)
display(stage_matrix)

## Progress Overview

In [ ]:
split_stage_counts = (
    processing_counts.groupby(["dataset", "model", "split", "stage"], observed=False, as_index=False)
    .first()
)

for model in sorted(split_stage_counts["model"].unique()):
    model_counts = split_stage_counts[split_stage_counts["model"] == model]
    progress_plot = sns.catplot(
        data=model_counts,
        kind="bar",
        x="stage",
        y="processed_texts",
        hue="split",
        col="dataset",
        col_wrap=4,
        height=3.2,
        aspect=1.5,
        sharey=False,
    )
    progress_plot.set_xticklabels(rotation=45, ha="right")
    progress_plot.set_axis_labels("Stage", "Processed texts")
    progress_plot.fig.suptitle(f"Progress by dataset: {model}", y=1.02)
    plt.show()
    plt.close(progress_plot.fig)

stage2_counts = processing_counts[processing_counts["stage"] == "token_similarity"]
for model in sorted(stage2_counts["model"].unique()):
    model_stage2 = stage2_counts[stage2_counts["model"] == model]
    stage2_plot = sns.catplot(
        data=model_stage2,
        kind="bar",
        x="dataset",
        y="processed_texts",
        hue="split",
        height=3.8,
        aspect=1.5,
        sharey=False,
    )
    stage2_plot.set_xticklabels(rotation=45, ha="right")
    stage2_plot.set_axis_labels("Dataset", "Stage 2 processed texts")
    stage2_plot.fig.suptitle(f"Stage 2 processed texts by dataset: {model}", y=1.02)
    plt.show()
    plt.close(stage2_plot.fig)